In [ ]:
dbutils.widgets.text("ticker", "AAPL")
dbutils.widgets.text("mic_code", "XNAS")
dbutils.widgets.text("analysis_date", "2024-02-07")

ticker = dbutils.widgets.get("ticker")
mic_code = dbutils.widgets.get("mic_code")
analysis_date = dbutils.widgets.get("analysis_date")

## Transaction Cost Analysis (TCA) using BMLL

Transaction Cost Analysis (TCA) helps traders and portfolio managers evaluate execution quality by measuring the costs incurred when executing trades. This notebook demonstrates a practical TCA workflow using BMLL's high-quality market data.

**Key Metrics:**
- **Arrival Price**: The mid-price at the time of trade execution
- **Implementation Shortfall**: The difference between arrival price and execution price
- **Market Impact**: Price movement after trade execution (markouts)
- **Price Improvement**: Trading inside the spread

**Important**: The BMLL `trades_plus` table contains ALL market trades. For TCA, you must identify YOUR specific executions.

### 1. Load Your Execution Data

**Choose ONE of these methods to identify your trades:**

#### Option A: Upload Execution File
Upload a CSV/Excel file with your fills. Required columns:
- `ticker`, `timestamp`, `price`, `quantity`, `side` (buy/sell)

#### Option B: Filter by Trade ID
If you know your broker's Trade IDs, filter BMLL data by TradeId

#### Option C: Use Sample Data (Demo)
Create synthetic executions for demonstration purposes

In [ ]:
# Import necessary libraries — Databricks (Spark) variant
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from datetime import datetime, timedelta

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: '%.6f' % x)

In [ ]:
# ==================== CONFIGURATION ====================
# Choose your method: 'UPLOAD' or 'SAMPLE'
METHOD = 'SAMPLE'  # Change this!

# Common parameters
TICKER = "AAPL"
MIC_CODE = "XNAS"  # Primary exchange MIC (used for reference/labelling)
MIC_TRADES = "@SIP"  # For US equities BMLL aggregates trades onto the SIP tape
ANALYSIS_DATE = "2024-02-07"

# Method-specific parameters
EXECUTION_FILE_PATH = "/dbfs/tmp/my_executions.csv"  # For UPLOAD method

print(f"Method selected: {METHOD}")
print(f"Analyzing: {TICKER} on {ANALYSIS_DATE}")

### Method A: Upload Your Execution File

In [ ]:
if METHOD == 'UPLOAD':
    # Load your execution file from DBFS
    # Expected format: ticker, timestamp, price, quantity, side
    # timestamp should be in UTC

    my_executions = pd.read_csv(EXECUTION_FILE_PATH)
    my_executions.columns = my_executions.columns.str.lower()
    my_executions['timestamp'] = pd.to_datetime(my_executions['timestamp'])

    my_executions_spark = spark.createDataFrame(my_executions)
    my_executions_spark.createOrReplaceTempView("my_executions_temp")

    print(f"Loaded {len(my_executions):,} execution records")
    display(my_executions.head())

### Method B: Create Sample Executions (Demo)

Generates ~30 synthetic execution records spread across the trading day.
Execution prices will be derived from real BMLL bid/ask data so TCA metrics are realistic.

In [ ]:
if METHOD == 'SAMPLE':
    # Create synthetic execution data for demonstration
    np.random.seed(42)

    # US equity hours: 09:30-16:00 ET = 14:30-21:00 UTC
    base_time = pd.Timestamp(f"{ANALYSIS_DATE} 14:30:00")

    sample_executions = []
    for i in range(30):
        minutes_offset = np.random.randint(0, 390)  # 6.5 hours = 390 minutes
        exec_time = base_time + timedelta(minutes=minutes_offset)
        sample_executions.append({
            'exec_id': f'EXEC_{i+1:03d}',
            'ticker': TICKER,
            'timestamp': exec_time,
            'side': np.random.choice(['buy', 'sell']),
            'quantity': int(np.random.randint(100, 5000)),
            'price': None,  # filled from market data below
        })

    my_executions = pd.DataFrame(sample_executions).sort_values('timestamp').reset_index(drop=True)

    print(f"Created {len(my_executions)} sample executions for {TICKER} on {ANALYSIS_DATE}")
    print("\nNote: These are SYNTHETIC trades for demonstration purposes.")
    print("For real TCA, use Method A (upload file) or provide broker Trade IDs.\n")
    display(my_executions[['exec_id', 'timestamp', 'side', 'quantity']])

## 2. Enrich with Market Data from BMLL

Match each execution to the nearest preceding market quote from BMLL to obtain:
- Best bid / ask at execution time
- Post-trade mid-price snapshots (markouts)
- Market spread and conditions

### 2a. Quick data preview

In [ ]:
# Preview the available BMLL Trades Plus data
spark.sql(f"""SELECT `TRADE_TIMESTAMP`, `PRICE`, `BEST_BID_PRICE_AT_PRIMARY`, `BEST_ASK_PRICE_AT_PRIMARY`, (`BEST_BID_PRICE_AT_PRIMARY` + `BEST_ASK_PRICE_AT_PRIMARY`) / 2 AS `MID_PRICE`, `BMLL_TRADE_TYPE`, `AGGRESSOR_SIDE` FROM `market_data`.`vw_trades_plus` WHERE `TICKER` = '{TICKER}' AND `MIC` = '{MIC_TRADES}' AND `TRADE_DATE` = '{ANALYSIS_DATE}' AND `BMLL_TRADE_TYPE` IN ('LIT', 'OTC') ORDER BY `TRADE_TIMESTAMP` NULLS LAST LIMIT 20""").display()

### 2b. ASOF join executions with BMLL market data (Databricks)

Load all required market quote columns from TRADES_PLUS, then do a single ASOF join per
execution path (SAMPLE or UPLOAD) using the Spark DataFrame API.
No large `.toPandas()` calls on market data — all processing runs in Spark.

In [ ]:
# Collect one representative market quote per minute — safe to pull to driver.
# For a 6.5-hour trading day this gives at most ~390 rows regardless of tick density.
# We then do the ASOF join in pandas before broadcasting back to Spark.
_market_quotes_pd = spark.sql(f"""SELECT DATE_TRUNC('MINUTE', `TRADE_TIMESTAMP`) AS `snap_ts`, FIRST(`BEST_BID_PRICE_AT_PRIMARY`) AS `BestBid`, FIRST(`BEST_ASK_PRICE_AT_PRIMARY`) AS `BestAsk`, FIRST((`BEST_BID_PRICE_AT_PRIMARY` + `BEST_ASK_PRICE_AT_PRIMARY`) / 2) AS `MarketMid`, FIRST(`BEST_ASK_PRICE_AT_PRIMARY` - `BEST_BID_PRICE_AT_PRIMARY`) AS `Spread`, FIRST(`POST_TRADE_MID_100MS_AT_PRIMARY`) AS `PostMid100ms`, FIRST(`POST_TRADE_MID_1000MS_AT_PRIMARY`) AS `PostMid1000ms`, FIRST(`POST_TRADE_MID_5000MS_AT_PRIMARY`) AS `PostMid5000ms`, FIRST(`POST_TRADE_MID_30000MS_AT_PRIMARY`) AS `PostMid30000ms`, FIRST(`POST_TRADE_MID_60000MS_AT_PRIMARY`) AS `PostMid60000ms` FROM `market_data`.`vw_trades_plus` WHERE `TICKER` = '{TICKER}' AND `MIC` = '{MIC_TRADES}' AND `TRADE_DATE` = '{ANALYSIS_DATE}' AND `BMLL_TRADE_TYPE` IN ('LIT', 'OTC') GROUP BY DATE_TRUNC('MINUTE', `TRADE_TIMESTAMP`) ORDER BY `snap_ts` NULLS LAST""").toPandas()  # at most ~390 rows — safe to collect

print(f"Market snapshot rows (1-min buckets): {len(_market_quotes_pd)}")
if len(_market_quotes_pd) == 0:
    raise ValueError(f"No market data found for {TICKER} {MIC_TRADES} on {ANALYSIS_DATE}. "
                     "Check that TICKER, MIC_TRADES, and ANALYSIS_DATE are correct.")

_market_quotes_pd['snap_ts'] = pd.to_datetime(_market_quotes_pd['snap_ts'])

In [ ]:
# Build execution DataFrame (SAMPLE or UPLOAD)
if METHOD == 'SAMPLE':
    _exec_pd = my_executions[['exec_id', 'ticker', 'timestamp', 'side', 'quantity']].copy()
elif METHOD == 'UPLOAD':
    _exec_pd = my_executions.copy()
else:
    raise ValueError(f"Unknown METHOD: {METHOD}")

_exec_pd['timestamp'] = pd.to_datetime(_exec_pd['timestamp'])

# pandas ASOF join: for each execution find the most recent preceding market snapshot
_exec_sorted   = _exec_pd.sort_values('timestamp')
_market_sorted = _market_quotes_pd.sort_values('snap_ts')

_enriched_pd = pd.merge_asof(
    _exec_sorted,
    _market_sorted,
    left_on='timestamp',
    right_on='snap_ts',
    direction='backward',
).drop(columns=['snap_ts'], errors='ignore')

# For SAMPLE, derive execution price from bid/ask (simulate 10% price improvement inside spread)
if METHOD == 'SAMPLE' or 'price' not in _enriched_pd.columns or _enriched_pd['price'].isna().all():
    _enriched_pd['price'] = np.where(
        _enriched_pd['side'] == 'buy',
        _enriched_pd['BestAsk'] - (_enriched_pd['BestAsk'] - _enriched_pd['BestBid']) * 0.1,
        np.where(
            _enriched_pd['side'] == 'sell',
            _enriched_pd['BestBid'] + (_enriched_pd['BestAsk'] - _enriched_pd['BestBid']) * 0.1,
            _enriched_pd['MarketMid'],
        )
    )

print(f"Matched {len(_enriched_pd)} executions with market data")
print(f"Null prices: {_enriched_pd['price'].isna().sum()}")

# Convert enriched pandas DF back to Spark for distributed TCA computation
enriched_df = spark.createDataFrame(_enriched_pd)
display(_enriched_pd[['exec_id', 'timestamp', 'side', 'quantity', 'price',
                       'MarketMid', 'BestBid', 'BestAsk', 'Spread']].head(10))

## 3. Calculate TCA Metrics (Databricks)

Now that we have your executions matched with market data, calculate costs and impacts
using Spark transformations — all in distributed memory.

In [ ]:
# Calculate TCA metrics in Spark
tca_df = enriched_df.withColumn(
    "ArrivalCost",
    F.when(F.col("side") == "buy",  F.col("price") - F.col("MarketMid"))
     .when(F.col("side") == "sell", F.col("MarketMid") - F.col("price"))
     .otherwise(0)
).withColumn(
    "ArrivalCostBps",
    (F.col("ArrivalCost") / F.col("MarketMid")) * 10000
).withColumn(
    "SpreadBps",
    (F.col("Spread") / F.col("MarketMid")) * 10000
).withColumn(
    "HalfSpreadBps",
    F.col("SpreadBps") / 2
).withColumn(
    "PriceImprovement",
    F.when(F.col("side") == "buy",  F.col("BestAsk") - F.col("price"))
     .when(F.col("side") == "sell", F.col("price")   - F.col("BestBid"))
     .otherwise(0)
).withColumn(
    "PriceImprovementBps",
    (F.col("PriceImprovement") / F.col("MarketMid")) * 10000
).withColumn(
    "Notional",
    F.col("price") * F.col("quantity")
)

# Markouts at each horizon
_horizons = [
    ("100ms",   "PostMid100ms"),
    ("1000ms",  "PostMid1000ms"),
    ("5000ms",  "PostMid5000ms"),
    ("30000ms", "PostMid30000ms"),
    ("60000ms", "PostMid60000ms"),
]
for lbl, col in _horizons:
    tca_df = tca_df.withColumn(
        f"Markout_{lbl}",
        F.when(F.col("side") == "buy",
               F.col(col) - F.col("price"))
         .when(F.col("side") == "sell",
               F.col("price") - F.col(col))
         .otherwise(None)
    ).withColumn(
        f"Markout_{lbl}Bps",
        (F.col(f"Markout_{lbl}") / F.col("MarketMid")) * 10000
    )

# Collect only the small enriched set (~30 rows) to pandas for visualisation
tca_pdf = tca_df.toPandas()

print(f"\nTCA analysis ready for {len(tca_pdf)} executions")
print(f"Total notional: ${tca_pdf['Notional'].sum():,.2f}\n")
display(tca_pdf[['exec_id', 'timestamp', 'side', 'quantity', 'price',
                 'MarketMid', 'ArrivalCostBps', 'PriceImprovementBps']].head(10))

## 4. TCA Summary Report

In [ ]:
# Compute volume-weighted aggregate TCA metrics entirely in Spark, then collect the tiny result
agg_df = tca_df.agg(
    F.count("exec_id").alias("total_executions"),
    F.sum("Notional").alias("total_notional"),
    F.sum(F.when(F.col("side") == "buy",  1).otherwise(0)).alias("buys"),
    F.sum(F.when(F.col("side") == "sell", 1).otherwise(0)).alias("sells"),
    F.mean("Notional").alias("avg_notional"),
    (F.sum(F.col("SpreadBps")        * F.col("Notional")) / F.sum("Notional")).alias("vw_spread_bps"),
    (F.sum(F.col("HalfSpreadBps")    * F.col("Notional")) / F.sum("Notional")).alias("vw_half_spread_bps"),
    (F.sum(F.col("ArrivalCostBps")   * F.col("Notional")) / F.sum("Notional")).alias("vw_arrival_cost_bps"),
    (F.sum(F.col("PriceImprovementBps") * F.col("Notional")) / F.sum("Notional")).alias("vw_pi_bps"),
    F.sum(F.when(F.col("PriceImprovement") > 0, 1).otherwise(0)).alias("n_improved"),
    (F.sum(F.col("Markout_100msBps")   * F.col("Notional")) / F.sum("Notional")).alias("vw_mo_100ms"),
    (F.sum(F.col("Markout_1000msBps")  * F.col("Notional")) / F.sum("Notional")).alias("vw_mo_1000ms"),
    (F.sum(F.col("Markout_60000msBps") * F.col("Notional")) / F.sum("Notional")).alias("vw_mo_60000ms"),
)

agg = agg_df.collect()[0]

summary = pd.DataFrame({
    'Metric': [
        'Total Executions', 'Total Notional ($)', 'Buys', 'Sells',
        'Avg Execution Size ($)', '',
        'Avg Market Spread (bps)', 'Expected Cost (half spread, bps)',
        'Actual Arrival Cost (bps)', 'Cost vs Expectation (bps)', '',
        'Price Improvement (bps)', '% Executions with Improvement', '',
        'Market Impact @ 100ms (bps)', 'Market Impact @ 1s (bps)',
        'Market Impact @ 60s (bps)',
    ],
    'Value': [
        f"{agg['total_executions']:,}",
        f"${agg['total_notional']:,.2f}",
        f"{agg['buys']:,}",
        f"{agg['sells']:,}",
        f"${agg['avg_notional']:,.2f}",
        '',
        f"{agg['vw_spread_bps']:.2f}",
        f"{agg['vw_half_spread_bps']:.2f}",
        f"{agg['vw_arrival_cost_bps']:.2f}",
        f"{agg['vw_arrival_cost_bps'] - agg['vw_half_spread_bps']:.2f}",
        '',
        f"{agg['vw_pi_bps']:.2f}",
        f"{agg['n_improved'] / agg['total_executions'] * 100:.1f}%",
        '',
        f"{agg['vw_mo_100ms']:.2f}"   if agg['vw_mo_100ms']   is not None else 'N/A',
        f"{agg['vw_mo_1000ms']:.2f}"  if agg['vw_mo_1000ms']  is not None else 'N/A',
        f"{agg['vw_mo_60000ms']:.2f}" if agg['vw_mo_60000ms'] is not None else 'N/A',
    ]
})

print("\n" + "=" * 70)
print(f"TCA SUMMARY: {TICKER} - {ANALYSIS_DATE}")
print("=" * 70 + "\n")
display(summary)

print("\nInterpretation Guide:")
print("  Arrival Cost    : How much you paid vs mid price")
print("  Price Improvement: Savings from executing inside the spread")
print("  Markout (positive) = unfavorable — price moved against you after trade")
print("  Markout (negative) = favorable  — price moved in your favor")

## 5. Visualizations

In [ ]:
# ---- Databricks visualisation: collect only small aggregated results ----

# Markout curve (one scalar per horizon) — already in tca_pdf (~30 rows, safe)
markout_horizons = ['100ms', '1s', '5s', '30s', '60s']
markout_cols     = ['Markout_100msBps', 'Markout_1000msBps', 'Markout_5000msBps',
                    'Markout_30000msBps', 'Markout_60000msBps']
markout_values = []
for col in markout_cols:
    valid = tca_pdf[tca_pdf[col].notna()]
    markout_values.append(
        float(np.average(valid[col], weights=valid['Notional'])) if len(valid) > 0 else 0.0
    )

# Intraday stats — aggregate in Spark then collect (O(hours) rows)
tca_df_hour = tca_df.withColumn("Hour", F.hour(F.col("timestamp").cast("timestamp")))
intraday_agg = (
    tca_df_hour.groupBy("Hour")
    .agg(
        (F.sum(F.col("ArrivalCostBps") * F.col("Notional")) / F.sum("Notional")).alias("AvgCostBps"),
        F.count("exec_id").alias("Count"),
        F.sum("Notional").alias("Notional"),
    )
    .orderBy("Hour")
    .toPandas()  # small aggregated result — safe to collect
)

# 2x2 Dashboard
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# A. Arrival Cost Distribution
vw_cost = float(np.average(tca_pdf['ArrivalCostBps'], weights=tca_pdf['Notional']))
axes[0, 0].hist(tca_pdf['ArrivalCostBps'].dropna(), bins=15, edgecolor='black', alpha=0.7, color='steelblue')
axes[0, 0].axvline(vw_cost, color='red', linestyle='--', linewidth=2, label=f'VW avg ({vw_cost:.1f} bps)')
axes[0, 0].set_xlabel('Arrival Cost (bps)', fontsize=11)
axes[0, 0].set_ylabel('Number of Executions', fontsize=11)
axes[0, 0].set_title('A) Arrival Cost Distribution', fontsize=12, fontweight='bold')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# B. Markout Curve
axes[0, 1].plot(markout_horizons, markout_values, marker='o', linewidth=2, markersize=8, color='darkgreen')
axes[0, 1].axhline(y=0, color='red', linestyle='--', alpha=0.5)
axes[0, 1].set_xlabel('Time After Execution', fontsize=11)
axes[0, 1].set_ylabel('Avg Market Impact (bps)', fontsize=11)
axes[0, 1].set_title('B) Market Impact (Markout) Curve', fontsize=12, fontweight='bold')
axes[0, 1].grid(True, alpha=0.3)

# C. Intraday Arrival Cost by Hour
if not intraday_agg.empty:
    ax_c  = axes[1, 0]
    ax_c2 = ax_c.twinx()
    ax_c.bar(intraday_agg['Hour'], intraday_agg['AvgCostBps'], color='steelblue', alpha=0.7)
    ax_c2.plot(intraday_agg['Hour'], intraday_agg['Notional'],
               color='darkorange', marker='o', linewidth=1.5)
    ax_c.set_xlabel('Hour of Day (UTC)', fontsize=11)
    ax_c.set_ylabel('Avg Arrival Cost (bps)', fontsize=11, color='steelblue')
    ax_c2.set_ylabel('Total Notional ($)',     fontsize=11, color='darkorange')
    ax_c.set_title('C) Intraday Execution Quality by Hour', fontsize=12, fontweight='bold')
    ax_c.grid(True, alpha=0.3)

# D. Price Improvement Rate
improvement_pct    = (tca_pdf['PriceImprovement'].fillna(0) > 0).sum() / len(tca_pdf) * 100
no_improvement_pct = 100 - improvement_pct
axes[1, 1].pie(
    [improvement_pct, no_improvement_pct],
    labels=[f'Improved\n({improvement_pct:.1f}%)', f'No Improvement\n({no_improvement_pct:.1f}%)'],
    colors=['lightgreen', 'lightcoral'],
    startangle=90,
)
axes[1, 1].set_title('D) Price Improvement Rate', fontsize=12, fontweight='bold')

plt.suptitle(f'TCA Dashboard — {TICKER} {ANALYSIS_DATE}', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Intraday breakdown table
print("\nIntraday Execution Quality by Hour of Day:")
print("-" * 55)
display(intraday_agg.rename(columns={
    'Hour':       'Hour (UTC)',
    'AvgCostBps': 'VW Arrival Cost (bps)',
    'Count':      'Executions',
    'Notional':   'Total Notional ($)',
}))

## 6. Execution-Level Detail

Drill into specific executions for detailed analysis.

In [ ]:
# Show worst and best executions by arrival cost
display_cols = [c for c in ['exec_id', 'timestamp', 'side', 'quantity', 'price',
                             'MarketMid', 'ArrivalCostBps', 'PriceImprovementBps',
                             'Markout_60000msBps'] if c in tca_pdf.columns]

print("WORST EXECUTIONS (Highest Arrival Cost):")
display(tca_pdf.nlargest(5, 'ArrivalCostBps')[display_cols])

print("\nBEST EXECUTIONS (Lowest Arrival Cost):")
display(tca_pdf.nsmallest(5, 'ArrivalCostBps')[display_cols])